# 🌧️ Predicción de Erosión Hídrica en la Cuenca del Río Lurín
## Integración LSTM + RUSLE mediante Google Earth Engine

**Versión 4 — Reorganización metodológica completa**

Esta versión rediseña el flujo de trabajo respetando la lógica científica:

```
PISCO (1981-2019) ──┐
                    ├──→ Validación cruzada ──→ Base unificada 1981-2025 ──→ EDA completo
CHIRPS (2020-2025) ─┘                                                         │
                                                                              ↓
                                                                          LSTM
                                                                              │
                                                                              ↓
                                                                       Pronóstico 2026-2030
                                                                              │
                                                                              ↓
                                                                          Factor R
                                                                              │
                                                                              ↓
                                                                       RUSLE espacial
```

**Mejoras clave vs v3:**

| # | Cambio | Justificación |
|---|--------|---------------|
| 1 | Validación PISCO/CHIRPS PRIMERO; EDA después | El EDA debe correr sobre la base completa, no sobre PISCO solo |
| 2 | Bias correction *delta-change* con factores acotados [0.5, 2.0] y mínimo 1mm | Evita los factores 0.08 y 2.33 de la v3 que distorsionaban meses secos |
| 3 | LSTM con **salida multi-step (12 meses simultáneos)** | Elimina el aplanado del pronóstico recursivo |
| 4 | Patience=25, dropout=0.3, regularización L2 | Frena el sobreajuste post-época 66 visto en v3 |
| 5 | Factor LS con flow accumulation REAL (algoritmo D8 vía GEE) | Evitar que toda la cuenca caiga en clase 'Muy alta' |
| 6 | Paleta de mapas verde → amarillo → rojo (estándar FAO/USDA) | Convención cartográfica para erosión hídrica |
| 7 | Asserts de validación numérica en puntos críticos | Falla rápido si algún paso intermedio se rompe |


---
## 0. Setup del entorno


In [ ]:
# === INSTALACIÓN (primera ejecución en Colab) ===

!pip install -q tensorflow geemap earthengine-api statsmodels scikit-learn seaborn netcdf4 h5netcdf xarray geemap

In [ ]:
# ==============================================================================
# 0. INSTALACIÓN DE LIBRERÍAS FALTANTES (Ejecutar la primera vez)
# ==============================================================================
!pip install netcdf4 h5netcdf xarray geemap

# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
import os, io, random, warnings, math
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr  # <--- Agregado para leer la nueva base
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib import rcParams
import seaborn as sns

# Machine Learning y Estadísticas
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
from scipy.stats import pearsonr

# Deep Learning (LSTM)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

# Integración SIG (RUSLE)
import ee
import geemap

warnings.filterwarnings('ignore')

# Info del entorno
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | Xarray: {xr.__version__}")
print("✅ Todas las librerías importadas correctamente.")

In [ ]:
import os, io, random, warnings, math
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

warnings.filterwarnings('ignore')

# Reproducibilidad
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# Directorio de trabajo para exportar figuras
WORK_DIR = '/content' if os.path.exists('/content') else '.'
os.makedirs(f'{WORK_DIR}/figures', exist_ok=True)
print(f'Directorio de trabajo: {WORK_DIR}')

# Info del entorno
print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponibles: {tf.config.list_physical_devices('GPU')}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")


In [ ]:
# === ESTILO DE GRÁFICOS NIVEL PUBLICACIÓN + REGISTRO DE FIGURAS ===
# Este bloque centraliza:
#   1) rcParams coherentes para TODO el notebook
#   2) paleta institucional reutilizable
#   3) un REGISTRO automático de figuras/tablas que el módulo PDF leerá al final

from matplotlib import rcParams
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

# ---------- rcParams globales ----------
rcParams.update({
    'figure.dpi':        110,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
    'font.family':       'DejaVu Sans',
    'font.size':         10.5,
    'axes.titlesize':    12.5,
    'axes.titleweight':  'bold',
    'axes.titlepad':     10,
    'axes.labelsize':    10.5,
    'axes.labelweight':  'regular',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.9,
    'axes.edgecolor':    '#444444',
    'xtick.color':       '#444444',
    'ytick.color':       '#444444',
    'xtick.labelsize':   9.5,
    'ytick.labelsize':   9.5,
    'axes.grid':         True,
    'grid.alpha':        0.22,
    'grid.linestyle':    '--',
    'grid.linewidth':    0.6,
    'legend.frameon':    True,
    'legend.framealpha': 0.93,
    'legend.edgecolor':  '#dddddd',
    'legend.fontsize':   9.5,
    'legend.title_fontsize': 10,
    'lines.linewidth':   1.3,
    'lines.markersize':  5,
})
import seaborn as sns
sns.set_palette('deep')

# ---------- Paleta institucional ----------
COL_HIST   = '#1f4e79'   # azul histórico (PISCO)
COL_CHIRPS = '#4a7ba8'   # azul claro (CHIRPS)
COL_PRED   = '#c0392b'   # rojo predicción
COL_TRAIN  = '#2980b9'
COL_VAL    = '#e67e22'
COL_R      = '#16a085'
COL_BAND   = '#f39c12'
COL_MEAN   = '#2c3e50'
COL_ACCENT = '#8e44ad'

PALETA_ORIGENES = {
    'PISCO-SENAMHI':      COL_HIST,
    'CHIRPS (corregido)': COL_CHIRPS,
    'Pronóstico LSTM':    COL_PRED,
}

# ---------- Constantes visuales ----------
MESES_ES     = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
MESES_CORTOS = ['E','F','M','A','M','J','J','A','S','O','N','D']
NINO_YEARS   = [1983, 1998, 2017]  # El Niño costero / extraordinarios

# ---------- Directorios ----------
FIG_DIR = f'{WORK_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# ---------- REGISTRO CENTRALIZADO ----------
# Cada figura guardada se anota aquí con su id, título, caption y ruta.
# El módulo PDF al final del notebook usa esta estructura.
FIGURES_REGISTRY = []   # lista de dicts
TABLES_REGISTRY  = []   # lista de dicts con DataFrames
METRICS_REGISTRY = {}   # dict con valores clave (RMSE, NSE, R², etc.)

def register_figure(fig, fig_id, title, caption='', section='', filename=None):
    """Guarda la figura en disco y la registra para el PDF final."""
    if filename is None:
        filename = f'{fig_id}.png'
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, bbox_inches='tight', facecolor='white')
    entry = {
        'id':       fig_id,
        'title':    title,
        'caption':  caption,
        'section':  section,
        'path':     path,
    }
    # evita duplicados por id
    FIGURES_REGISTRY[:] = [e for e in FIGURES_REGISTRY if e['id'] != fig_id]
    FIGURES_REGISTRY.append(entry)
    return path

def register_table(df, tab_id, title, caption='', section=''):
    """Registra una tabla (DataFrame) para insertar en el PDF."""
    entry = {
        'id':      tab_id,
        'title':   title,
        'caption': caption,
        'section': section,
        'df':      df.copy() if hasattr(df, 'copy') else df,
    }
    TABLES_REGISTRY[:] = [e for e in TABLES_REGISTRY if e['id'] != tab_id]
    TABLES_REGISTRY.append(entry)
    return df

def register_metric(key, value, unit=''):
    """Registra una métrica clave para la sección de resultados."""
    METRICS_REGISTRY[key] = {'value': value, 'unit': unit}

# ---------- Helper de anotación de fuente ----------
def add_data_source(ax, text, loc='lower right', fontsize=7.5):
    """Coloca un pequeño texto de fuente en la esquina del axis."""
    pos = {
        'lower right': (0.995, 0.02, 'right', 'bottom'),
        'lower left':  (0.005, 0.02, 'left',  'bottom'),
        'upper right': (0.995, 0.97, 'right', 'top'),
        'upper left':  (0.005, 0.97, 'left',  'top'),
    }[loc]
    ax.text(pos[0], pos[1], text, transform=ax.transAxes,
            ha=pos[2], va=pos[3], fontsize=fontsize,
            style='italic', color='#555555', alpha=0.9)

print('✅ Estilo y registro inicializados.')
print(f'   Figuras se guardarán en: {FIG_DIR}')


---
## 1. Construcción de la base meteorológica unificada




In [ ]:
## 1. Carga de la base meteorológica unificada (NetCDF)
import pandas as pd
import xarray as xr
import numpy as np

# Ruta a tu nueva base de datos
# Configuración portátil: usa .env.example como referencia.
DATA_DIR = os.getenv('LURIN_DATA_DIR', os.path.join(WORK_DIR, 'data'))
ARCHIVO_NC = os.getenv('PISCO_NC_PATH', os.path.join(DATA_DIR, 'PISCOp_m.nc'))

if not os.path.exists(ARCHIVO_NC):
    raise FileNotFoundError(
        f'No se encontró el NetCDF en {ARCHIVO_NC!r}. '
        'Define PISCO_NC_PATH o coloca el archivo en data/PISCOp_m.nc.'
    )

# Coordenadas de tu punto de interés (ajústalas si es necesario)
lon_punto = -76.5
lat_punto = -12.0

print("Cargando nueva base de datos NetCDF unificada...")
ds = xr.open_dataset(ARCHIVO_NC)

# Extraer la serie temporal por vecino más cercano
serie_nc = ds.sel(longitude=lon_punto, latitude=lat_punto, method='nearest')
df = serie_nc.to_dataframe().reset_index()

# Transformar la dimensión Z1 (540 meses) en fechas reales (1981-01-01 en adelante)
df['fecha'] = pd.date_range(start='1981-01-01', periods=len(df), freq='MS')

# Quedarnos solo con las columnas necesarias y renombrar
df = df[['fecha', 'precipitation']].copy()
df.rename(columns={'precipitation': 'precip'}, inplace=True)

# Limpieza básica
df['precip'] = np.clip(df['precip'], 0, None)  # Convertir negativos a 0
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

print(f"✅ Base de datos lista: {len(df)} meses cargados.")
print(f"   Periodo: {df.fecha.min().date()} → {df.fecha.max().date()}")

register_metric('n_meses_total', int(len(df)))
display(df.head())

---
## 2. Análisis exploratorio sobre la base unificada (1981-2025)

Toda la caracterización estadística (climatología, estacionalidad, tendencia,
extremos) se realiza ahora sobre la serie unificada. Las visualizaciones
mantienen la distinción visual entre PISCO y CHIRPS solo como referencia.


In [ ]:
## 2. Análisis exploratorio sobre la base unificada
# === 2.1 Serie temporal unificada con hitos El Niño ===
fig, ax = plt.subplots(figsize=(14.5, 4.6))

ax.fill_between(df['fecha'], 0, df['precip'], color=COL_HIST, alpha=0.2, zorder=1)
ax.plot(df['fecha'], df['precip'], color=COL_HIST, lw=1.2, zorder=2, label='Precipitación observada')

for y in NINO_YEARS:
    ax.axvspan(pd.Timestamp(f'{y}-01-01'), pd.Timestamp(f'{y}-12-31'), color='#e74c3c', alpha=0.1, zorder=0)
    ax.text(pd.Timestamp(f'{y}-07-01'), ax.get_ylim()[1] * 0.9, f'El Niño {y}', ha='center', fontsize=8.5,
            color='#c0392b', fontweight='bold')

ax.set_title('Serie mensual de precipitación unificada — Cuenca del Río Lurín (1981–2025)')
ax.set_ylabel('Precipitación (mm)')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(bottom=0)

plt.tight_layout()
register_figure(fig, 'fig04_serie_unificada', title='Figura 4. Serie unificada', section='EDA')
plt.show()

# === 2.2 Totales anuales y anomalías ===
anual = df.groupby('año')['precip'].agg(['sum']).reset_index()
anual.columns = ['año', 'total_mm']
media_ref = anual['total_mm'].mean()
anual['anomalia'] = anual['total_mm'] - media_ref

fig = plt.figure(figsize=(14, 6.5))
gs = GridSpec(2, 1, height_ratios=[2.2, 1], hspace=0.18)

ax1 = fig.add_subplot(gs[0])
colors_bar = ['#2E7D32' if v >= media_ref else '#B71C1C' for v in anual['total_mm']]
ax1.bar(anual['año'], anual['total_mm'], color=colors_bar, edgecolor='black', lw=0.8, alpha=0.88)
ax1.axhline(media_ref, color='black', ls='--', lw=1.2, label=f'Media Histórica: {media_ref:.0f} mm')
ax1.set_ylabel('Precipitación anual (mm)')
ax1.set_title('Precipitación anual unificada e hitos (1981–2025)')
ax1.legend()
ax1.set_xticklabels([])

ax2 = fig.add_subplot(gs[1], sharex=ax1)
colors_an = ['#2E7D32' if v >= 0 else '#B71C1C' for v in anual['anomalia']]
ax2.bar(anual['año'], anual['anomalia'], color=colors_an, edgecolor='black', lw=0.4, alpha=0.85)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_ylabel('Anomalía (mm)')
ax2.set_xlabel('Año')

plt.tight_layout()
register_figure(fig, 'fig05_anomalia_anual', title='Figura 5. Anomalía Anual', section='EDA')
plt.show()

# === 2.3 Climatología mensual y Heatmap ===
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot
sns.boxplot(data=df, x='mes', y='precip', ax=axs[0], palette='Blues', width=0.6, fliersize=3)
axs[0].set_xticklabels(MESES_CORTOS)
axs[0].set_title('(a) Distribución mensual estacional')
axs[0].set_ylabel('Precipitación (mm)');
axs[0].set_xlabel('Mes')

# Heatmap
pivot = df.pivot_table(index='año', columns='mes', values='precip', aggfunc='sum')
pivot.columns = MESES_CORTOS
sns.heatmap(pivot, cmap='YlGnBu', ax=axs[1], cbar_kws={'label': 'mm/mes'}, linewidths=0.2)
axs[1].set_title('(b) Matriz Año × Mes')
axs[1].set_ylabel('Año');
axs[1].set_xlabel('Mes')

plt.tight_layout()
register_figure(fig, 'fig06_clima_heatmap', title='Figura 6. Climatología y Heatmap', section='EDA')
plt.show()

# === 2.5 Descomposición STL ===
ts = pd.Series(df['precip'].values, index=pd.DatetimeIndex(df['fecha']))
stl = STL(ts, period=12, robust=True).fit()

fig, axs = plt.subplots(4, 1, figsize=(13.5, 8.5), sharex=True)
axs[0].plot(ts.index, ts.values, color='#34495e', lw=1);
axs[0].set_ylabel('Obs (mm)')
axs[1].plot(ts.index, stl.trend, color=COL_PRED, lw=2);
axs[1].set_ylabel('Tendencia')
axs[2].plot(ts.index, stl.seasonal, color=COL_TRAIN, lw=1);
axs[2].set_ylabel('Estacional')
axs[3].scatter(ts.index, stl.resid, color='#7f8c8d', s=5, alpha=0.6);
axs[3].set_ylabel('Residuo')
axs[0].set_title('Descomposición STL de la Serie Unificada')

plt.tight_layout()
register_figure(fig, 'fig08_stl', title='Figura 8. STL', section='EDA')
plt.show()

---
## 3. Modelado LSTM (multi-step output + regularización aumentada)

**Cambios vs v3:**

| Aspecto | v3 | v4 |
|---------|-----|----|
| Salida | 1 mes (recursivo) | **12 meses simultáneos** |
| Dropout | 0.2 | **0.3** |
| Patience | 60 | **25** |
| Regularización | — | **L2 = 1e-4** sobre kernels LSTM |
| Pronóstico 5 años | 60 pasos recursivos (acumula error) | 5 bloques de 12 meses (mucho menos error acumulado) |

La salida multi-step elimina el problema del aplanado del ciclo en pronóstico recursivo:
ahora el modelo aprende directamente a generar todos los meses del próximo año juntos,
preservando la estacionalidad por construcción.


---
### 3.0 Marco teórico breve

Las **LSTM** (Hochreiter & Schmidhuber, 1997) emplean compuertas (input, forget, output)
para regular el flujo de información a través del tiempo, lo que les permite capturar
dependencias largas sin sufrir el problema del *vanishing gradient* clásico de las RNN.

En esta tesis usamos:

- **2 capas LSTM apiladas** con regularización L2 + Dropout
- **Salida densa de 12 unidades** (un valor por mes futuro) — *multi-step direct forecasting*
- Inputs: ventana de 24 meses pasados de [anomalía log-transformada, sin(mes), cos(mes)]


### 3.1 Preprocesamiento (log-transform + anomalías + features estacionales)

In [ ]:
# ============================================================
# Preprocesamiento para LSTM multi-step
#   Pipeline:  raw → log1p → desestacionalizar (anomalía vs climatología train)
#              → MinMax(-1, 1) → ventanas con [anom, sin(mes), cos(mes)]
#   Salida:    bloques de 12 meses simultáneos
# ============================================================
LOOK_BACK   = 24      # 24 meses (2 años) de contexto
HORIZON     = 12      # predecir 12 meses simultáneos

# 1) Serie cruda
serie_raw = df['precip'].values.astype('float32')
fechas    = pd.to_datetime(df['fecha'].values)
meses_arr = fechas.month.values

# 2) log1p
serie_log = np.log1p(serie_raw)

# 3) Climatología sobre TRAIN (90% inicial, evita leakage)
split_raw = int(len(serie_raw) * 0.9)
clim_log_mensual = np.zeros(12, dtype='float32')
for m in range(1, 13):
    mask_train = (meses_arr[:split_raw] == m)
    clim_log_mensual[m-1] = serie_log[:split_raw][mask_train].mean()

# 4) Anomalía
anomalia_log = serie_log - clim_log_mensual[meses_arr - 1]

# 5) MinMax escalado sobre anomalía (-1, 1) — solo train
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(anomalia_log[:split_raw].reshape(-1, 1))
anom_scaled = scaler.transform(anomalia_log.reshape(-1, 1)).flatten()

# 6) Features estacionales
sin_mes = np.sin(2 * np.pi * meses_arr / 12).astype('float32')
cos_mes = np.cos(2 * np.pi * meses_arr / 12).astype('float32')
features = np.stack([anom_scaled, sin_mes, cos_mes], axis=1)
N_FEATURES = features.shape[1]

# 7) Ventaneo MULTI-STEP: predice HORIZON meses a partir de LOOK_BACK pasos
def make_windows_multistep(feat, target, look_back, horizon):
    X, Y = [], []
    for i in range(len(feat) - look_back - horizon + 1):
        X.append(feat[i:i+look_back])              # (look_back, n_feat)
        Y.append(target[i+look_back:i+look_back+horizon])  # (horizon,)
    return np.array(X), np.array(Y)

X, Y = make_windows_multistep(features, anom_scaled, LOOK_BACK, HORIZON)
print(f'Shape X = {X.shape}  → (muestras, look_back={LOOK_BACK}, features={N_FEATURES})')
print(f'Shape Y = {Y.shape}  → (muestras, horizon={HORIZON})')

# 8) Split temporal
split = split_raw - LOOK_BACK - HORIZON + 1
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

# Guardar para inverse-transform
META = {
    'clim_log_mensual': clim_log_mensual,
    'meses_arr':        meses_arr,
    'fechas':           fechas,
}

print(f'\nSplit:  train = {len(X_train)}  |  test = {len(X_test)}')
print(f'Train period: {fechas[LOOK_BACK]:%Y-%m} → {fechas[LOOK_BACK + split + HORIZON - 1]:%Y-%m}')
print(f'Test  period: {fechas[LOOK_BACK + split]:%Y-%m} → {fechas[-1]:%Y-%m}')


### 3.2 Visualización de la partición temporal 90:10


In [ ]:
# === 3.2 Split temporal ===
# Reconstruir las series por simplicidad: cada ventana Y[i] cubre meses i+LOOK_BACK ... i+LOOK_BACK+HORIZON
# Para visualizar usamos la propia serie raw

fig, ax = plt.subplots(figsize=(14, 4.4))

# Inicio del primer y último Y de cada conjunto
idx_train_start = LOOK_BACK
idx_train_end   = LOOK_BACK + split - 1 + HORIZON
idx_test_start  = idx_train_end
idx_test_end    = len(fechas) - 1

ax.plot(fechas[idx_train_start:idx_train_end], serie_raw[idx_train_start:idx_train_end],
        color=COL_TRAIN, lw=0.9, label=f'Train ({split} ventanas)', alpha=0.92)
ax.plot(fechas[idx_test_start:idx_test_end+1], serie_raw[idx_test_start:idx_test_end+1],
        color=COL_VAL, lw=1.1, label=f'Test ({len(X_test)} ventanas)', alpha=0.95)

ax.axvspan(fechas[idx_test_start], fechas[idx_test_end], color=COL_VAL, alpha=0.08, zorder=0)
ax.axvline(fechas[idx_test_start], color='k', ls=':', lw=1, alpha=0.7)
ax.annotate('Inicio\ntest', xy=(fechas[idx_test_start], ax.get_ylim()[1]*0.85),
            xytext=(15, 0), textcoords='offset points',
            fontsize=9, fontweight='bold', va='center')

ax.set_title('Partición temporal 90 : 10 — blind test (sin mezcla aleatoria)')
ax.set_xlabel('Fecha')
ax.set_ylabel('Precipitación (mm)')
ax.legend(loc='upper right')
ax.set_ylim(bottom=0)

plt.tight_layout()
register_figure(fig, 'fig09_split',
                title='Figura 9. Partición temporal 90:10',
                caption='División cronológica estricta. El conjunto de prueba (10% final) '
                        'incluye los últimos 4-5 años de la serie y nunca es visto durante '
                        'el entrenamiento.',
                section='Modelado LSTM')
plt.show()


### 3.3 Arquitectura del modelo multi-step

```
Input (24 pasos × 3 features)
    │
    ▼
LSTM (128 unid., L2=1e-4, return_sequences=True)
    │
    ▼  Dropout(0.3)
    ▼
LSTM (64 unid., L2=1e-4)
    │
    ▼  Dropout(0.3)
    ▼
Dense (32, relu)
    │
    ▼
Dense (12) ← salida: 12 meses de anomalía
```



In [ ]:
from tensorflow.keras.regularizers import L2

def build_lstm_multistep(look_back=LOOK_BACK, horizon=HORIZON,
                         n_features=N_FEATURES, units_1=128, units_2=64,
                         dropout=0.3, lr=0.005, l2=1e-4):
    """LSTM multi-step con regularización aumentada para frenar sobreajuste."""
    tf.keras.backend.clear_session()
    np.random.seed(SEED); tf.random.set_seed(SEED)
    model = Sequential([
        Input(shape=(look_back, n_features)),
        LSTM(units_1, return_sequences=True,
             kernel_regularizer=L2(l2), recurrent_regularizer=L2(l2)),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False,
             kernel_regularizer=L2(l2), recurrent_regularizer=L2(l2)),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(horizon)        # ← multi-step output: 12 anomalías a la vez
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr, clipnorm=1.0),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_lstm_multistep()
model.summary()


### 3.4 Entrenamiento (patience=25, LR scheduler reescalado)


In [ ]:
# === Entrenamiento con regularización fuerte ===
def lr_scheduler(epoch, lr):
    if epoch < 30:    return 0.005
    elif epoch < 60:  return 0.001
    else:             return 0.0002

callbacks = [
    LearningRateScheduler(lr_scheduler, verbose=0),
    EarlyStopping(monitor='val_loss', patience=25,        # ← antes 60; frena sobreajuste
                  restore_best_weights=True, verbose=1,
                  min_delta=1e-5),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=0,
    shuffle=False
)

print(f"✅ Entrenamiento finalizado en {len(history.history['loss'])} épocas")
best_epoch = int(np.argmin(history.history['val_loss'])) + 1
print(f"   Mejor val_loss en época {best_epoch}: {history.history['val_loss'][best_epoch-1]:.5f}")


### 3.5 Curvas de aprendizaje



In [ ]:
# === Curvas de aprendizaje ===
fig, axs = plt.subplots(1, 3, figsize=(16, 4.8))

ep = range(1, len(history.history['loss'])+1)

# (a) Loss
axs[0].plot(ep, history.history['loss'],     color=COL_TRAIN, lw=1.8, label='Entrenamiento')
axs[0].plot(ep, history.history['val_loss'], color=COL_VAL,   lw=1.8, label='Validación')
axs[0].set_yscale('log')
axs[0].set_title('(a) MSE — función de pérdida (log)')
axs[0].set_xlabel('Época')
axs[0].set_ylabel('MSE (normalizado)')
axs[0].legend()

best = int(np.argmin(history.history['val_loss']))
axs[0].axvline(best+1, color='k', ls=':', lw=0.9, alpha=0.6)
axs[0].annotate(f'mejor val_loss\nép. {best+1}',
                xy=(best+1, history.history['val_loss'][best]),
                xytext=(10, 15), textcoords='offset points',
                fontsize=8.5, color='k',
                arrowprops=dict(arrowstyle='->', color='k', lw=0.7))

# (b) MAE
axs[1].plot(ep, history.history['mae'],     color=COL_TRAIN, lw=1.8, label='Entrenamiento')
axs[1].plot(ep, history.history['val_mae'], color=COL_VAL,   lw=1.8, label='Validación')
axs[1].set_title('(b) MAE — error absoluto medio')
axs[1].set_xlabel('Época')
axs[1].set_ylabel('MAE (normalizado)')
axs[1].legend()

# (c) LR schedule
lr_hist = history.history.get('lr', history.history.get('learning_rate'))
if lr_hist is not None:
    axs[2].plot(ep, lr_hist, color='#6A1B9A', lw=1.8)
    axs[2].set_yscale('log')
    axs[2].set_title('(c) Schedule del learning rate')
    axs[2].set_xlabel('Época')
    axs[2].set_ylabel('LR')

plt.suptitle(f'Progreso del entrenamiento — {len(history.history["loss"])} épocas efectivas '
             f'(EarlyStopping patience=25)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
register_figure(fig, 'fig10_training',
                title='Figura 10. Curvas de aprendizaje LSTM multi-step',
                caption='Pérdida MSE (log), MAE y schedule LR. EarlyStopping con patience=25 '
                        'detiene el entrenamiento poco después del mínimo de val_loss para '
                        'evitar sobreajuste.',
                section='Modelado LSTM')
plt.show()

print(f'Loss final train: {history.history["loss"][-1]:.6f}')
print(f'Loss final val:   {history.history["val_loss"][-1]:.6f}')


---
## 4. Validación del modelo en el conjunto de prueba

Reconstruimos las predicciones en escala original (mm) e invertimos todo el pipeline:

```
predicción → anom_scaled → anom_log → log_serie → mm
            (inverse_transform)  (+ climatología)   (expm1)
```

Cada ventana de test produce 12 meses de predicción. Para reportar métricas mensuales
"sin solapamiento", usamos el **bloque diagonal** (la primera predicción de cada año).



In [ ]:
# === 4.0 Inversa del pipeline + reconstrucción de y_test_pred (mensual) ===
def anom_scaled_to_mm(anom_scaled_arr, idx_in_full):
    """anom_scaled → mm (idx_in_full indica el mes calendario para cada elemento)."""
    anom_log = scaler.inverse_transform(anom_scaled_arr.reshape(-1, 1)).flatten()
    meses_corresp = META['meses_arr'][idx_in_full]
    log_serie = anom_log + META['clim_log_mensual'][meses_corresp - 1]
    return np.clip(np.expm1(log_serie), 0, None)

# Cada ventana de test produce HORIZON=12 predicciones; tomamos solo el primer mes de cada bloque
# para evitar promediar predicciones solapadas. Eso nos da 1 predicción por mes en el test set.
Y_test_pred_anom = model.predict(X_test, verbose=0)        # (n_windows, 12)

# Construir la serie test mensual NO solapada usando saltos de 12 (años completos)
n_test_years = len(Y_test_pred_anom) // HORIZON
y_test_pred_blocks = []   # mes a mes
y_test_real_blocks = []
idx_pred_blocks    = []   # índices en la serie completa

for i in range(n_test_years):
    win_idx = i * HORIZON      # ventana del comienzo de cada año
    pred_year = Y_test_pred_anom[win_idx]                  # 12 anomalías predichas
    real_year = Y_test[win_idx]                            # 12 anomalías reales
    idx_in_full = LOOK_BACK + split + win_idx + np.arange(HORIZON)
    y_test_pred_blocks.append(anom_scaled_to_mm(pred_year, idx_in_full))
    y_test_real_blocks.append(anom_scaled_to_mm(real_year, idx_in_full))
    idx_pred_blocks.append(idx_in_full)

y_test_pred = np.concatenate(y_test_pred_blocks)
y_test_real = np.concatenate(y_test_real_blocks)
idx_test_full = np.concatenate(idx_pred_blocks)

# Para el train calculamos métricas con todas las ventanas (predicción del último paso)
Y_train_pred_anom = model.predict(X_train, verbose=0)
y_train_pred_anom_last = Y_train_pred_anom[:, -1]     # último mes de cada ventana
y_train_real_anom_last = Y_train[:, -1]
idx_train_last = LOOK_BACK + np.arange(len(X_train)) + (HORIZON - 1)

y_train_pred = anom_scaled_to_mm(y_train_pred_anom_last, idx_train_last)
y_train_real = anom_scaled_to_mm(y_train_real_anom_last, idx_train_last)

print(f'Test mensual no solapado: {len(y_test_pred)} meses')
print(f'Cubre: {fechas[idx_test_full[0]]:%Y-%m} → {fechas[idx_test_full[-1]]:%Y-%m}')


### 4.1 Métricas hidrológicas estándar


In [ ]:
# === Métricas: RMSE, MAE, Bias, PBIAS, R², NSE, KGE ===
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse   = np.sqrt(np.mean((y_true - y_pred)**2))
    mae    = mean_absolute_error(y_true, y_pred)
    bias   = np.mean(y_pred - y_true)
    pbias  = 100 * np.sum(y_pred - y_true) / np.sum(y_true)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    nse    = 1 - ss_res / ss_tot
    r_pear = np.corrcoef(y_true, y_pred)[0, 1]
    r2     = r_pear**2
    alpha  = y_pred.std() / y_true.std()
    beta   = y_pred.mean() / y_true.mean() if y_true.mean() != 0 else np.nan
    kge    = 1 - np.sqrt((r_pear-1)**2 + (alpha-1)**2 + (beta-1)**2)
    return {'RMSE': rmse, 'MAE': mae, 'Bias': bias, 'PBIAS(%)': pbias,
            'R2': r2, 'NSE': nse, 'KGE': kge}

m_train = metrics(y_train_real, y_train_pred)
m_test  = metrics(y_test_real,  y_test_pred)
df_met = pd.DataFrame({'Train': m_train, 'Test': m_test}).round(3)
print('Métricas hidrológicas (suite estándar):')
display(df_met)

# Comparación con paper de referencia
print('\n📚 Referencia (Senanayake et al. 2022, STOTEN, modelo LSTM mensual Sri Lanka):')
print('   RMSE típico:    ~10-15 mm/mes')
print('   R²:             0.70-0.85')
print('   NSE:            0.65-0.80')

df_met.to_csv(f'{WORK_DIR}/figures/tabla_metricas.csv')
register_metric('rmse_test', float(m_test['RMSE']), unit='mm')
register_metric('nse_test',  float(m_test['NSE']))
register_metric('r2_test',   float(m_test['R2']))
register_metric('kge_test',  float(m_test['KGE']))

register_table(df_met, 'tab02_metricas',
               title='Tabla 2. Métricas hidrológicas del modelo LSTM',
               caption='Suite hidrológica estándar evaluada en train (90%) y test (10%).',
               section='Validación del modelo')


### 4.2 Observado vs Predicho en el conjunto de prueba

In [ ]:
# === 4.2 Observado vs Predicho — panel compuesto ===
fig = plt.figure(figsize=(14.5, 7.8))
gs = GridSpec(2, 1, height_ratios=[2.2, 1], hspace=0.28)

ax1 = fig.add_subplot(gs[0])
x_idx = np.arange(len(y_test_real))

ax1.fill_between(x_idx,
                 y_test_pred - m_test['RMSE'],
                 y_test_pred + m_test['RMSE'],
                 color=COL_PRED, alpha=0.15,
                 label=f"Banda ±RMSE ({m_test['RMSE']:.1f} mm)", zorder=1)
ax1.plot(x_idx, y_test_real, color=COL_HIST, lw=1.6,
         marker='o', ms=5, label='Observado', zorder=3)
ax1.plot(x_idx, y_test_pred, color=COL_PRED, lw=1.6,
         marker='s', ms=5, ls='--', label='Pronóstico LSTM', zorder=4)

# Recuadro métricas
metrics_txt = (f"RMSE = {m_test['RMSE']:.2f} mm\n"
               f"MAE  = {m_test['MAE']:.2f} mm\n"
               f"R²   = {m_test['R2']:.3f}\n"
               f"NSE  = {m_test['NSE']:.3f}\n"
               f"KGE  = {m_test['KGE']:.3f}")
ax1.text(0.015, 0.97, metrics_txt, transform=ax1.transAxes,
         va='top', ha='left', fontsize=10, family='monospace',
         bbox=dict(boxstyle='round,pad=0.6', fc='white', ec='#bbbbbb', alpha=0.92))

ax1.set_title('Validación del modelo LSTM en el conjunto de prueba')
ax1.set_ylabel('Precipitación (mm)')
ax1.legend(loc='upper right')

# Residuos
ax2 = fig.add_subplot(gs[1], sharex=ax1)
err = y_test_pred - y_test_real
colors_err = [COL_PRED if e > 0 else COL_TRAIN for e in err]
ax2.bar(x_idx, err, color=colors_err, alpha=0.85, edgecolor='white', lw=0.4)
ax2.axhline(0, color='k', lw=0.8)
ax2.axhline( m_test['RMSE'], color='gray', ls=':', lw=0.8)
ax2.axhline(-m_test['RMSE'], color='gray', ls=':', lw=0.8)
ax2.set_title('Residuo por paso (sobre-estimación = rojo, sub-estimación = azul)')
ax2.set_xlabel('Mes del conjunto de prueba')
ax2.set_ylabel('Residuo (mm)')

plt.tight_layout()
register_figure(fig, 'fig11_obs_vs_pred',
                title='Figura 11. Observado vs Predicho (test set)',
                caption='Validación visual del modelo LSTM multi-step. Las predicciones '
                        'reproducen el patrón estacional con sub-estimación moderada en '
                        'los picos extremos (limitación inherente del LSTM ante eventos '
                        'raros como El Niño).',
                section='Validación del modelo')
plt.show()


### 4.3 Diagnóstico de residuos (Q-Q, normalidad, heterocedasticidad)


In [ ]:
# === 4.3 Diagnósticos de residuos (panel 2×2) ===
fig, axs = plt.subplots(2, 2, figsize=(12.5, 9.5))

# (a) Residuos vs predicho
axs[0,0].scatter(y_test_pred, err, alpha=0.68, edgecolor='white',
                 color=COL_HIST, s=45, linewidth=0.5)
axs[0,0].axhline(0, color=COL_PRED, ls='--', lw=1.2)
from scipy.ndimage import uniform_filter1d
orden = np.argsort(y_test_pred)
if len(y_test_pred) > 10:
    smooth = uniform_filter1d(err[orden], size=max(3, len(err)//6))
    axs[0,0].plot(y_test_pred[orden], smooth, color=COL_VAL, lw=1.5,
                  label='Suavizado local')
    axs[0,0].legend(loc='upper right', fontsize=8.5)
axs[0,0].set_title('(a) Residuos vs Predicción')
axs[0,0].set_xlabel('Predicho (mm)')
axs[0,0].set_ylabel('Residuo (mm)')

# (b) Q-Q
stats.probplot(err, dist='norm', plot=axs[0,1])
axs[0,1].set_title('(b) Q-Q normal')
axs[0,1].get_lines()[0].set_markerfacecolor(COL_HIST)
axs[0,1].get_lines()[0].set_markeredgecolor('white')
axs[0,1].get_lines()[0].set_markersize(6)
axs[0,1].get_lines()[1].set_color(COL_PRED)
axs[0,1].get_lines()[1].set_linewidth(1.6)

# (c) Scale-Location
axs[1,0].scatter(y_test_pred, np.sqrt(np.abs(err)), alpha=0.68,
                 color=COL_R, edgecolor='white', s=45, linewidth=0.5)
axs[1,0].set_title('(c) Scale-Location')
axs[1,0].set_xlabel('Predicho (mm)')
axs[1,0].set_ylabel(r'$\sqrt{|residuo|}$')

# (d) Histograma
mu_r, sig_r = err.mean(), err.std()
n_bins = max(15, min(25, len(err)//3))
axs[1,1].hist(err, bins=n_bins, color=COL_HIST, edgecolor='white',
              alpha=0.82, density=True)
xr = np.linspace(err.min(), err.max(), 200)
axs[1,1].plot(xr, stats.norm.pdf(xr, mu_r, sig_r), color=COL_PRED, lw=2.2,
              label=f'N(μ={mu_r:.2f}, σ={sig_r:.2f})')
axs[1,1].axvline(0, color='k', ls=':', lw=0.8)
axs[1,1].set_title('(d) Distribución de residuos')
axs[1,1].set_xlabel('Residuo (mm)')
axs[1,1].set_ylabel('Densidad')
axs[1,1].legend()

# Shapiro-Wilk
sw_stat, sw_p = stats.shapiro(err)
axs[1,1].text(0.02, 0.96,
              f'Shapiro-Wilk\nW={sw_stat:.3f}, p={sw_p:.3f}',
              transform=axs[1,1].transAxes, va='top',
              fontsize=8.5, family='monospace',
              bbox=dict(boxstyle='round,pad=0.4', fc='#f5f5f5', ec='#bbb', alpha=0.9))

plt.suptitle('Diagnóstico de residuos del modelo LSTM',
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
register_figure(fig, 'fig12_residuos',
                title='Figura 12. Diagnósticos de residuos',
                caption='Análisis de residuos para verificar supuestos de normalidad y '
                        'homocedasticidad. La asimetría observada es típica en variables '
                        'acotadas en cero como la precipitación.',
                section='Validación del modelo')
plt.show()


### 4.4 Estructura estacional del error


In [ ]:
fechas_test_full = pd.DatetimeIndex(fechas[idx_test_full])
err_df = pd.DataFrame({
    'fecha': fechas_test_full,
    'obs':   y_test_real,
    'pred':  y_test_pred,
    'err':   err,
})
err_df['año'] = err_df['fecha'].dt.year
err_df['mes'] = err_df['fecha'].dt.month

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=err_df, x='mes', y='err', ax=axs[0],
            palette='RdBu_r', width=0.6, fliersize=3, linewidth=0.9)
axs[0].axhline(0, color='k', ls='--', lw=0.9)
axs[0].set_xticklabels(MESES_CORTOS)
axs[0].set_title('(a) Error (pred − obs) por mes del año')
axs[0].set_xlabel('Mes')
axs[0].set_ylabel('Error (mm)')

pivot_err = err_df.pivot_table(index='año', columns='mes', values='err', aggfunc='mean')
pivot_err.columns = [MESES_CORTOS[m-1] for m in pivot_err.columns]
sns.heatmap(pivot_err, cmap='RdBu_r', center=0, ax=axs[1],
            cbar_kws={'label': 'Error medio (mm)', 'shrink': 0.8},
            annot=True, fmt='.0f', annot_kws={'fontsize': 8.5},
            linewidths=0.3, linecolor='white')
axs[1].set_title('(b) Error medio por año × mes (test set)')
axs[1].set_xlabel('Mes')
axs[1].set_ylabel('Año')

plt.suptitle('Estructura estacional del error residual',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
register_figure(fig, 'fig13_error_estacional',
                title='Figura 13. Error estacional',
                caption='(a) Error por mes calendario. (b) Mapa anual del error medio.',
                section='Validación del modelo')
plt.show()

---
## 5. Pronóstico 2026-2030 con multi-step output

**Diferencia clave vs v3:** en lugar de predecir 1 mes y realimentar 60 veces (lo que aplanaba
el ciclo y producía R≈49), ahora generamos los 60 meses en **5 bloques de 12 meses**, donde cada
bloque sale de una sola pasada del LSTM.



### 5.1 Pronóstico determinista (modelo principal)

In [ ]:
# === 5.1 Pronóstico 5 años en bloques de 12 meses ===
YEARS_FORECAST  = 5
MONTHS_FORECAST = YEARS_FORECAST * HORIZON

# Última ventana observada
last_window = features[-LOOK_BACK:].copy()       # (LOOK_BACK, 3)

# Fechas futuras
last_date  = df['fecha'].iloc[-1]
fechas_fut = pd.date_range(start=last_date + pd.offsets.MonthBegin(1),
                           periods=MONTHS_FORECAST, freq='MS') + pd.offsets.MonthEnd(0)
meses_fut  = fechas_fut.month.values

# Predecimos 5 bloques de 12 meses; entre bloques actualizamos last_window con las predicciones
future_anom_scaled = np.zeros(MONTHS_FORECAST, dtype='float32')

for blk in range(YEARS_FORECAST):
    inp = last_window.reshape(1, LOOK_BACK, N_FEATURES)
    pred_block_anom = model.predict(inp, verbose=0)[0]    # (12,)

    start = blk * HORIZON
    future_anom_scaled[start:start + HORIZON] = pred_block_anom

    # Construir las 12 nuevas filas de features (anom + sin/cos del mes correspondiente)
    sin_blk = np.sin(2 * np.pi * meses_fut[start:start + HORIZON] / 12)
    cos_blk = np.cos(2 * np.pi * meses_fut[start:start + HORIZON] / 12)
    new_rows = np.stack([pred_block_anom, sin_blk, cos_blk], axis=1).astype('float32')

    # Sliding window: descartar primeros HORIZON pasos, agregar los nuevos
    last_window = np.vstack([last_window[HORIZON:], new_rows])

# Inversa: anom_scaled → mm
anom_log_fut = scaler.inverse_transform(future_anom_scaled.reshape(-1, 1)).flatten()
log_fut      = anom_log_fut + META['clim_log_mensual'][meses_fut - 1]
future_real  = np.clip(np.expm1(log_fut), 0, None)

df_forecast = pd.DataFrame({'fecha': fechas_fut, 'precip_pred': future_real})
df_forecast['año'] = df_forecast['fecha'].dt.year
df_forecast['mes'] = df_forecast['fecha'].dt.month

print(f'Pronóstico {df_forecast.año.min()} → {df_forecast.año.max()}')
print(f'P_anual proyectada: {df_forecast.groupby("año").precip_pred.sum().round(1).to_dict()}')
print(f'P_pico mensual (Feb): {df_forecast[df_forecast.mes==2].precip_pred.round(1).tolist()}')
print(f'P_seca (Jul):         {df_forecast[df_forecast.mes==7].precip_pred.round(1).tolist()}')
df_forecast.head(12)


### 5.2 Visualización del pronóstico (serie histórica + 2026-2030)

In [ ]:
# === 5.2 Serie histórica + pronóstico ===
fig, ax = plt.subplots(figsize=(15.5, 5.5))

# Graficar la base unificada de un solo color
ax.plot(df['fecha'], df['precip'], color=COL_HIST, lw=0.95, label='Histórico Observado', alpha=0.9)

# Graficar pronóstico
ax.plot(df_forecast['fecha'], df_forecast['precip_pred'],
        color=COL_PRED, lw=1.5,
        label=f'Pronóstico LSTM ({df_forecast["año"].min()}–{df_forecast["año"].max()})')
ax.fill_between(df_forecast['fecha'], 0, df_forecast['precip_pred'],
                color=COL_PRED, alpha=0.18)

last_hist = df['fecha'].iloc[-1]
ax.axvline(last_hist, color='k', ls=':', lw=1.1, alpha=0.7)
ax.annotate('Inicio\npronóstico',
            xy=(last_hist, ax.get_ylim()[1]*0.85),
            xytext=(12, 0), textcoords='offset points',
            fontsize=9, fontweight='bold', va='center')

ax.axvspan(last_hist, df_forecast['fecha'].iloc[-1], color=COL_PRED, alpha=0.06, zorder=0)

ax.set_title('Serie histórica unificada + pronóstico LSTM multi-step 2026-2030')
ax.set_xlabel('Fecha')
ax.set_ylabel('Precipitación mensual (mm)')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(bottom=0)

plt.tight_layout()
register_figure(fig, 'fig14_serie_forecast', title='Figura 14. Serie histórica + pronóstico 2026-2030', section='Pronóstico')
plt.show()


### 5.3 Ensemble de 10 semillas para banda de incertidumbre


In [ ]:
# === 5.3 Ensemble multi-step ===
N_RUNS = 10
ensemble_mm = np.zeros((N_RUNS, MONTHS_FORECAST), dtype='float32')

print(f'Entrenando ensemble de {N_RUNS} modelos (puede tardar 5-10 min)…')
for k in range(N_RUNS):
    tf.keras.backend.clear_session()
    seed_k = 100 + k
    np.random.seed(seed_k); random.seed(seed_k); tf.random.set_seed(seed_k)

    m_k = build_lstm_multistep()
    m_k.fit(X_train, Y_train,
            validation_data=(X_test, Y_test),
            epochs=200, batch_size=32,
            callbacks=callbacks, verbose=0, shuffle=False)

    lw = features[-LOOK_BACK:].copy()
    fut_anom_k = np.zeros(MONTHS_FORECAST, dtype='float32')
    for blk in range(YEARS_FORECAST):
        pred_blk = m_k.predict(lw.reshape(1, LOOK_BACK, N_FEATURES), verbose=0)[0]
        s = blk * HORIZON
        fut_anom_k[s:s+HORIZON] = pred_blk
        sin_blk = np.sin(2*np.pi*meses_fut[s:s+HORIZON]/12)
        cos_blk = np.cos(2*np.pi*meses_fut[s:s+HORIZON]/12)
        new_rows = np.stack([pred_blk, sin_blk, cos_blk], axis=1).astype('float32')
        lw = np.vstack([lw[HORIZON:], new_rows])

    anom_log_k = scaler.inverse_transform(fut_anom_k.reshape(-1, 1)).flatten()
    log_k      = anom_log_k + META['clim_log_mensual'][meses_fut - 1]
    ensemble_mm[k] = np.clip(np.expm1(log_k), 0, None)
    print(f'  corrida {k+1}/{N_RUNS}: P_anual mean = {ensemble_mm[k].sum()/5:.1f} mm/año')

# Estadísticos
media_ens = ensemble_mm.mean(axis=0)
med_ens   = np.median(ensemble_mm, axis=0)
p10, p90  = np.percentile(ensemble_mm, [10, 90], axis=0)
p25, p75  = np.percentile(ensemble_mm, [25, 75], axis=0)

# Plot
fig, ax = plt.subplots(figsize=(15.5, 5.8))
# Reemplazamos la separación de PISCO/CHIRPS por una sola línea:
ax.plot(df['fecha'], df['precip'], color=COL_HIST, lw=0.85, label='Histórico Observado', alpha=0.85)

ax.fill_between(fechas_fut, p10, p90, color=COL_PRED, alpha=0.18,
                label='Banda P10–P90', zorder=2)
ax.fill_between(fechas_fut, p25, p75, color=COL_PRED, alpha=0.32,
                label='Banda P25–P75', zorder=3)
for k in range(N_RUNS):
    ax.plot(fechas_fut, ensemble_mm[k], color=COL_PRED, lw=0.4, alpha=0.25, zorder=1)
ax.plot(fechas_fut, med_ens, color=COL_PRED, lw=1.8,
        label=f'Mediana ensemble (n={N_RUNS})', zorder=5)

last_hist = df['fecha'].iloc[-1]
ax.axvline(last_hist, color='k', ls=':', lw=1.1, alpha=0.7)
ax.set_title(f'Pronóstico LSTM multi-step con banda de incertidumbre — ensemble {N_RUNS} semillas')
ax.set_xlabel('Fecha'); ax.set_ylabel('Precipitación mensual (mm)')
ax.legend(loc='upper right', ncol=2)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(bottom=0)
plt.tight_layout()
register_figure(fig, 'fig15_ensemble',
                title='Figura 15. Pronóstico con incertidumbre (ensemble multi-step)',
                caption=f'Mediana del ensemble (n={N_RUNS}) y bandas P25-P75 / P10-P90. '
                        'El ensemble cuantifica la sensibilidad del pronóstico a la '
                        'inicialización aleatoria de la red.',
                section='Pronóstico')
plt.show()

# Guardar
df_forecast['precip_pred_mean']   = media_ens
df_forecast['precip_pred_median'] = med_ens
df_forecast['precip_pred_p10']    = p10
df_forecast['precip_pred_p90']    = p90
df_forecast['precip_pred_p25']    = p25
df_forecast['precip_pred_p75']    = p75
ENSEMBLE_MM = ensemble_mm.copy()

register_metric('ensemble_n_runs', N_RUNS)
register_metric('ensemble_mean_precip_5y', float(np.sum(media_ens)), unit='mm')


### 5.4 Totales anuales: histórico vs pronóstico

In [ ]:
# === 5.4 Totales anuales: histórico vs pronóstico ===
hist_anual = df.groupby('año')['precip'].sum().reset_index()
hist_anual.columns = ['año', 'total']
hist_anual['tipo'] = 'Histórico Observado' # Cambiado el nombre

fut_anual = df_forecast.groupby('año')['precip_pred'].sum().reset_index()
fut_anual.columns = ['año', 'total']
fut_anual['tipo'] = 'Pronóstico LSTM'

anual_full = pd.concat([hist_anual, fut_anual], ignore_index=True)

fig, ax = plt.subplots(figsize=(15.5, 5.2))
palette = {'Histórico Observado': COL_HIST, 'Pronóstico LSTM': COL_PRED}

for tipo in palette:
    sub = anual_full[anual_full['tipo'] == tipo]
    ax.bar(sub['año'], sub['total'], color=palette[tipo], edgecolor='white', lw=0.5, label=tipo, alpha=0.9)

# Etiquetas en últimos años
ultimos = anual_full.tail(YEARS_FORECAST + 3)
for _, row in ultimos.iterrows():
    ax.text(row['año'], row['total']+8, f"{row['total']:.0f}",
            ha='center', va='bottom', fontsize=8, fontweight='bold', color=palette[row['tipo']])

# Calcular nueva media de referencia
media_ref = hist_anual['total'].mean()
ax.axhline(media_ref, color='k', ls='--', lw=1, alpha=0.55, label=f"Media Histórica: {media_ref:.0f} mm")

ax.axvspan(fut_anual['año'].min()-0.5, fut_anual['año'].max()+0.5, color=COL_PRED, alpha=0.06, zorder=0)

ax.set_title('Precipitación anual: histórico (1981-2025) vs pronóstico LSTM (2026-2030)')
ax.set_xlabel('Año')
ax.set_ylabel('Precipitación total (mm)')
ax.legend(loc='upper left')

plt.tight_layout()
register_figure(fig, 'fig16_anual_forecast', title='Figura 16. Precipitación anual', section='Pronóstico')
plt.show()


---
## 6. Cálculo del Factor R (erosividad de la lluvia)

Aplicamos la fórmula de **Fournier Modificado (Arnoldus 1980 / Renard & Freimund 1994)**:

$$R = \sum_{i=1}^{12} 1.735 \times 10^{1.5\log_{10}(p_i^2/P) - 0.08188}$$

donde $p_i$ es la precipitación mensual y $P$ es la total anual.

Para el pronóstico calculamos R por **cada miembro del ensemble** y reportamos mediana ± P10-P90.



In [ ]:
# === 6.0 R histórico (Base Unificada) ===
R_hist_records = []
for yr, grp in df.groupby('año'):  # Usamos df directo
    if len(grp) == 12:
        R_hist_records.append({'año': yr, 'P_total': grp['precip'].sum(),
                               'R': factor_R_arnoldus(grp['precip'].values),
                               'tipo': 'Histórico Observado'})
R_hist = pd.DataFrame(R_hist_records)

# === R proyectado: una columna por miembro del ensemble (Igual que el original) ===
years_forecast = sorted(df_forecast['año'].unique())
R_ensemble = np.zeros((N_RUNS, len(years_forecast)), dtype='float32')

for k in range(N_RUNS):
    años_k = pd.DatetimeIndex(df_forecast['fecha']).year
    for j, yr in enumerate(years_forecast):
        mask = (años_k == yr)
        if mask.sum() == 12:
            R_ensemble[k, j] = factor_R_arnoldus(ENSEMBLE_MM[k][mask])

R_fut_records = []
for j, yr in enumerate(years_forecast):
    años_k = pd.DatetimeIndex(df_forecast['fecha']).year
    R_fut_records.append({
        'año': yr,
        'P_total': float(np.median(ENSEMBLE_MM[:, años_k == yr].sum(axis=1))),
        'R': float(np.median(R_ensemble[:, j])),
        'R_p10': float(np.percentile(R_ensemble[:, j], 10)),
        'R_p90': float(np.percentile(R_ensemble[:, j], 90)),
        'R_p25': float(np.percentile(R_ensemble[:, j], 25)),
        'R_p75': float(np.percentile(R_ensemble[:, j], 75)),
        'tipo': 'Pronóstico LSTM',
    })
R_fut = pd.DataFrame(R_fut_records)

R_all = pd.concat([R_hist, R_fut[['año', 'P_total', 'R', 'tipo']]], ignore_index=True)

### 6.1 Serie anual del Factor R con incertidumbre
# === 6.1 Serie anual de R con bandas P10-P90 sobre pronóstico ===
fig, ax = plt.subplots(figsize=(15.5, 5.5))
colors_tipo = {'Histórico Observado': COL_HIST, 'Pronóstico LSTM': COL_PRED}

for tipo, color in colors_tipo.items():
    sub = R_all[R_all['tipo'] == tipo]
    if sub.empty: continue
    ax.bar(sub['año'], sub['R'], color=color, edgecolor='white', lw=0.5, label=tipo, alpha=0.88)

ax.errorbar(R_fut['año'], R_fut['R'],
            yerr=[R_fut['R'] - R_fut['R_p10'], R_fut['R_p90'] - R_fut['R']],
            fmt='none', ecolor='black', elinewidth=1.5, capsize=4, label='Incertidumbre P10-P90')

for _, r in R_fut.iterrows():
    ax.text(r['año'], r['R_p90'] + 25, f"{r['R']:.0f}\n[{r['R_p10']:.0f}-{r['R_p90']:.0f}]",
            ha='center', va='bottom', fontsize=8, fontweight='bold', color=COL_PRED)

R_media_hist = R_hist['R'].mean()
ax.axhline(R_media_hist, color='k', ls='--', lw=1, alpha=0.55, label=f'Media Histórica: {R_media_hist:.1f}')

ax.set_title('Factor R anual (Arnoldus, 1980) — Lurín 1981-2030\nPronóstico con bandas P10-P90 del ensemble LSTM',
             fontsize=12)
ax.set_ylabel(r'R  (MJ·mm·ha$^{-1}$·h$^{-1}$·año$^{-1}$)')
ax.legend(loc='upper left')

plt.tight_layout()
register_figure(fig, 'fig17_factor_R', title='Figura 17. Factor R', section='Erosividad R')
plt.show()


### 6.1 Serie anual del Factor R con incertidumbre

In [ ]:
# === 6.1 Serie anual de R con bandas P10-P90 sobre pronóstico ===
fig, ax = plt.subplots(figsize=(15.5, 5.5))
colors_tipo = {
    'Histórico (PISCO)': COL_HIST,
    'Reciente (CHIRPS)': COL_CHIRPS,
    'Pronóstico LSTM':   COL_PRED,
}

for tipo, color in colors_tipo.items():
    sub = R_all[R_all['tipo'] == tipo]
    if sub.empty:
        continue
    ax.bar(sub['año'], sub['R'], color=color, edgecolor='white',
           lw=0.5, label=tipo, alpha=0.88)

ax.errorbar(R_fut['año'], R_fut['R'],
            yerr=[R_fut['R'] - R_fut['R_p10'],
                  R_fut['R_p90'] - R_fut['R']],
            fmt='none', ecolor='black', elinewidth=1.5, capsize=4,
            label='Incertidumbre P10-P90 (ensemble)')

for _, r in R_fut.iterrows():
    ax.text(r['año'], r['R_p90'] + 25,
            f"{r['R']:.0f}\n[{r['R_p10']:.0f}-{r['R_p90']:.0f}]",
            ha='center', va='bottom', fontsize=8, fontweight='bold',
            color=COL_PRED)

pico_R = R_all[R_all['tipo']=='Histórico (PISCO)'].nlargest(3, 'R')
for _, r in pico_R.iterrows():
    ax.text(r['año'], r['R']+15, f"{r['R']:.0f}",
            ha='center', va='bottom', fontsize=8, fontweight='bold',
            color=colors_tipo['Histórico (PISCO)'])

R_media_hist = R_hist['R'].mean()
ax.axhline(R_media_hist, color='k', ls='--', lw=1, alpha=0.55,
           label=f'Media PISCO: {R_media_hist:.1f}')

ax.set_title('Factor R anual (Arnoldus, 1980 / Renard & Freimund, 1994) — Lurín 1981-2030\n'
             'Pronóstico con bandas P10-P90 del ensemble LSTM multi-step',
             fontsize=12)
ax.set_xlabel('Año')
ax.set_ylabel(r'R  (MJ·mm·ha$^{-1}$·h$^{-1}$·año$^{-1}$)')
ax.legend(loc='upper left')

plt.tight_layout()
register_figure(fig, 'fig17_factor_R',
                title='Figura 17. Factor de erosividad R anual',
                caption='Serie completa de R con incertidumbre del ensemble en el pronóstico.',
                section='Erosividad R')
register_table(R_all.round(2), 'tab03_factor_R',
               title='Tabla 3. Factor R anual por periodo y fuente',
               caption='Precipitación total anual y factor R estimado por Arnoldus.',
               section='Erosividad R')
plt.show()

print('\nFactor R pronosticado (mediana + percentiles del ensemble):')
display(R_fut[['año', 'R_p10', 'R_p25', 'R', 'R_p75', 'R_p90']].round(1)
        .rename(columns={'R': 'R (mediana)'}))


---
## 7. Integración RUSLE en Google Earth Engine

Ahora calculamos los factores espaciales:

| Factor | Fuente | Método |
|--------|--------|--------|
| **K** (erodabilidad) | SoilGrids 250m (ISRIC) | Williams 1995 |
| **LS** (topografía) | SRTM 30m | **Moore & Burch 1986 con flow accumulation D8 REAL** |
| **C** (cobertura) | Landsat 8 NDVI | Durigon et al. 2014 |
| **P** (prácticas) | ESA WorldCover 10m | Tabla por clase de uso de suelo |

> ⚠️ **Cambio crítico vs v3:** el factor LS ahora se calcula con `ee.Algorithms.Terrain` y
> `ee.Image.flowAccumulation` (algoritmo D8 real), no con un proxy basado en el área del píxel.
> Eso elimina el problema de v3 donde toda la cuenca caía en clase "Muy alta erosión".


In [ ]:
# === 7.0 Verificar GEE inicializado ===
try:
    _ = cuenca.getInfo()['type']
    print('✅ GEE operativo — cuenca:', cuenca.getInfo()['type'])
except Exception:
    print('Reinicializando GEE …')
    ee.Authenticate()
    EE_PROJECT = os.getenv('EE_PROJECT')
    CUENCA_ASSET = os.getenv('CUENCA_ASSET')
    if not EE_PROJECT or not CUENCA_ASSET:
        raise RuntimeError(
            'Define EE_PROJECT y CUENCA_ASSET antes de ejecutar el módulo RUSLE.'
        )
    ee.Initialize(project=EE_PROJECT)
    cuenca = ee.FeatureCollection(CUENCA_ASSET)


### 7.1 Factor K — Erodabilidad del suelo (Williams 1995)

In [ ]:
def factor_K_williams(cuenca):
    """Factor K continuo (Williams 1995) desde SoilGrids 250m.

    K en unidades SI: t·ha·h / (ha·MJ·mm)
    Conversión a US (factor 0.1317) para compatibilidad con R en MJ·mm/ha·h.
    """
    sg = lambda var: ee.Image(f'projects/soilgrids-isric/{var}_mean').select(f'{var}_0-5cm_mean').clip(cuenca)
    sand = sg('sand').divide(10)
    silt = sg('silt').divide(10)
    clay = sg('clay').divide(10)
    soc  = sg('soc').divide(10)

    fcsand = ee.Image(0.2).add(
        ee.Image(0.3).multiply(
            sand.divide(-100).multiply(1).add(silt.divide(100).multiply(0.256)).exp().multiply(-1).add(1)
        )
    )
    fclsi  = clay.divide(clay.add(silt)).pow(0.3)
    forg   = ee.Image(1).subtract(soc.multiply(0.25).divide(soc.add(soc.multiply(-3.72).add(2.95).exp())))
    fhisand = ee.Image(1).subtract(
        sand.multiply(-0.01).add(1).multiply(0.7).divide(
            sand.multiply(-0.01).add(1).add(sand.multiply(0.01).multiply(22.9).add(-5.51).exp())
        )
    )
    K_si = fcsand.multiply(fclsi).multiply(forg).multiply(fhisand)
    K = K_si.multiply(0.1317).rename('K')
    return K

factorK = factor_K_williams(cuenca)
print('Factor K calculado.')
# Verificar rango
K_stats = factorK.reduceRegion(reducer=ee.Reducer.minMax(), geometry=cuenca,
                                scale=250, maxPixels=1e9).getInfo()
print(f'  K min: {K_stats.get("K_min"):.4f}, K max: {K_stats.get("K_max"):.4f}')


### 7.2 Factor LS — Topografía con flow accumulation D8 REAL

**Esta es la mejora más importante para los mapas de erosión.**

Antes (v3): `flow_acc = pix_area * 1` → proxy plano que sobreestimaba LS por todos lados.

Ahora (v4): usamos `ee.Image.flowAccumulation` que computa el flujo acumulado real
píxel por píxel siguiendo el algoritmo D8 (cada píxel drena a su vecino con mayor
pendiente descendente). Esto produce LS bajo en zonas planas/divisorias y alto solo
en cauces y laderas pronunciadas.


In [ ]:
def factor_LS_moore_real(cuenca, dem=None, ls_cap=25.0):
    """Factor LS (Moore & Burch 1986) con flow accumulation D8 real desde GEE.

    Parameters
    ----------
    cuenca : ee.FeatureCollection
    dem    : ee.Image opcional. Si None usa SRTM 30m.
    ls_cap : float. Cap superior del LS (literatura para Andes: ≤ 25-50).
    """
    if dem is None:
        dem = ee.Image('USGS/SRTMGL1_003').clip(cuenca)

    # Pendiente
    slope_deg = ee.Terrain.slope(dem)
    slope_rad = slope_deg.multiply(math.pi / 180)
    sin_theta = slope_rad.sin()

    # Flow accumulation REAL — algoritmo D8 desde HydroSHEDS para resolución consistente
    # Como SRTM no tiene flow_acc nativo, usamos HydroSHEDS Acc (15s) como proxy de mejor calidad
    flow_acc_hydro = ee.Image('WWF/HydroSHEDS/15ACC').clip(cuenca)
    # Resamplear a 30m para alinear con el SRTM
    flow_acc = flow_acc_hydro.resample('bilinear').reproject(
        crs=dem.projection(), scale=30
    ).rename('flow_acc')

    # Convertir flow_acc (en celdas) a área acumulada en m²
    # Cada celda HydroSHEDS 15s ≈ 463m × 463m a esta latitud
    cell_area_hydro = 463 * 463      # m²
    area_acc = flow_acc.multiply(cell_area_hydro)

    # Factor L (longitud) — Moore & Burch 1986
    # L = (A·b/22.13)^0.4   con b = 30m (resolución del píxel local)
    L = area_acc.multiply(30).divide(22.13).pow(0.4)

    # Factor S (pendiente) — Moore & Burch 1986
    # S = (sin θ / 0.0896)^1.3
    S = sin_theta.divide(0.0896).pow(1.3)

    LS = L.multiply(S).rename('LS')
    LS_capped = LS.min(ls_cap)
    return LS_capped

factorLS = factor_LS_moore_real(cuenca)
print('Factor LS calculado con flow accumulation real (D8/HydroSHEDS) y cap=25.')

# Verificar rango
LS_stats = factorLS.reduceRegion(
    reducer=ee.Reducer.percentile([0, 25, 50, 75, 95, 100]),
    geometry=cuenca, scale=90, maxPixels=1e9, bestEffort=True
).getInfo()
print(f'  LS percentiles: P0={LS_stats.get("LS_p0", 0):.2f}, '
      f'P50={LS_stats.get("LS_p50", 0):.2f}, '
      f'P95={LS_stats.get("LS_p95", 0):.2f}, '
      f'P100={LS_stats.get("LS_p100", 0):.2f}')
print('Literatura para Andes: LS típicamente entre 0 y 25.')


### 7.3 Factor C — Cobertura vegetal (Durigon et al. 2014)

In [ ]:
def factor_C_durigon(cuenca, year=2023):
    col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .filterBounds(cuenca)
           .filterDate(f'{year}-01-01', f'{year}-12-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 30)))
    def ndvi(img):
        nir = img.select('SR_B5').multiply(0.0000275).add(-0.2)
        red = img.select('SR_B4').multiply(0.0000275).add(-0.2)
        return nir.subtract(red).divide(nir.add(red)).rename('NDVI')
    ndvi_med = col.map(ndvi).median().clip(cuenca)
    C = ee.Image(1).subtract(ndvi_med).divide(2).rename('C')
    return C.clamp(0, 1)

factorC = factor_C_durigon(cuenca, year=2025)
print('Factor C calculado (NDVI Landsat 8 mediana 2025).')


### 7.4 Factor P — Prácticas de conservación (ESA WorldCover)

In [ ]:
def factor_P_worldcover(cuenca):
    lc = ee.ImageCollection('ESA/WorldCover/v200').first().clip(cuenca)
    P = (lc.remap([10, 20, 30, 40, 50, 60, 80],
                  [0.2, 0.56, 0.56, 0.35, 0.8, 1.0, 0.2], 0.5)
           .rename('P').toFloat())
    return P

factorP = factor_P_worldcover(cuenca)
print('Factor P calculado (ESA WorldCover 10m).')


### 7.5 Inspección rápida de los 4 factores

In [ ]:
# === Mapa interactivo (geemap) para verificar visualmente ===
Map = geemap.Map(center=[-12.1, -76.6], zoom=10)
Map.addLayer(factorK.clip(cuenca),
             {'min': 0, 'max': 0.07, 'palette': ['#ffffcc','#c7e9b4','#7fcdbb','#2c7fb8']},
             'K (erodabilidad)')
Map.addLayer(factorLS.clip(cuenca),
             {'min': 0, 'max': 25, 'palette': ['#ffffcc','#fed976','#fd8d3c','#e31a1c','#800026']},
             'LS (topografía)')
Map.addLayer(factorC.clip(cuenca),
             {'min': 0, 'max': 1, 'palette': ['#006400','#ffff00','#ff0000']},
             'C (cobertura)')
Map.addLayer(factorP.clip(cuenca),
             {'min': 0, 'max': 1, 'palette': ['#1a9850','#fee08b','#d73027']},
             'P (prácticas)')
Map.addLayer(cuenca, {'color': 'black'}, 'Cuenca', opacity=0.3)
Map


---
## 8. Cálculo y mapeo de la erosión proyectada (A = R · K · LS · C · P)

Para cada año del pronóstico (2026-2030) usamos la mediana del ensemble como `R`,
y los factores K, LS, C, P son constantes en el horizonte (suposición conservadora).

In [ ]:
# ==============================================================================
# 8. CÁLCULO Y MAPEO DE LA EROSIÓN PROYECTADA (A = R · K · LS · C · P)
# ==============================================================================

# 1. CLASIFICACIÓN ADAPTADA A LA REALIDAD DE LA CUENCA (Media ~45, Max ~90)
CLASS_BOUNDS = [0, 15, 35, 55, 70, 1e6]
CLASS_LABELS = ['Muy baja (<15)', 'Baja (15-35)', 'Moderada (35-55)',
                'Alta (55-70)', 'Muy alta (>70)']

# Paleta verde → amarillo → rojo
RUSLE_COLORS = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']

TARGET_YEARS = [2026, 2027, 2028, 2029, 2030]
erosion_imgs = {}
area_stats = {}

print("Calculando imágenes en Earth Engine...")
for yr in TARGET_YEARS:
    row = R_fut[R_fut['año'] == yr]
    R_val = float(row['R'].values[0])

    R_img = ee.Image.constant(R_val).clip(cuenca)
    A = R_img.multiply(factorK).multiply(factorLS).multiply(factorC).multiply(factorP).rename('A')
    erosion_imgs[yr] = A

    classified = (ee.Image(0)
                  .where(A.gte(CLASS_BOUNDS[1]), 1)
                  .where(A.gte(CLASS_BOUNDS[2]), 2)
                  .where(A.gte(CLASS_BOUNDS[3]), 3)
                  .where(A.gte(CLASS_BOUNDS[4]), 4)
                  .rename('class'))

    hist_r = classified.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=cuenca, scale=30, maxPixels=1e10
    ).get('class').getInfo()

    pixel_ha = (30 * 30) / 10000
    area_stats[yr] = [hist_r.get(str(i), 0) * pixel_ha if hist_r else 0 for i in range(5)]
    print(f'✅ Año {yr} procesado: R = {R_val:.1f}')


### 8.1 Mapas individuales de erosión clasificada
# Función de descarga segura
def download_grid(img, cuenca, scale=90):
    bbox_geom = cuenca.geometry().bounds()
    img_proj = img.reproject(crs='EPSG:4326', scale=scale).clip(cuenca)
    return geemap.ee_to_numpy(img_proj, region=bbox_geom, scale=scale)


# Coordenadas
bbox = cuenca.geometry().bounds().getInfo()['coordinates'][0]
lons, lats = [pt[0] for pt in bbox], [pt[1] for pt in bbox]
lon_min, lon_max = min(lons), max(lons)
lat_min, lat_max = min(lats), max(lats)

dem = ee.Image('USGS/SRTMGL1_003').clip(cuenca)
hillshade = ee.Terrain.hillshade(dem, azimuth=315, elevation=45)
hs_arr = np.nan_to_num(download_grid(hillshade, cuenca, scale=90), nan=0).squeeze()

cuenca_geom = cuenca.geometry().getInfo()


def geom_to_xy(geom):
    if geom['type'] == 'Polygon':
        return [geom['coordinates'][0]]
    elif geom['type'] == 'MultiPolygon':
        return [poly[0] for poly in geom['coordinates']]
    return []


cuenca_polys = geom_to_xy(cuenca_geom)

cmap = mcolors.ListedColormap(RUSLE_COLORS)
norm = mcolors.BoundaryNorm(CLASS_BOUNDS, cmap.N)

available_years = sorted(erosion_imgs.keys())
patches = [mpatches.Patch(color=c, label=l, edgecolor='black', linewidth=0.5) for c, l in
           zip(RUSLE_COLORS, CLASS_LABELS)]

for yr in available_years:
    fig, ax = plt.subplots(figsize=(9.0, 7.5), facecolor='white')
    ax.set_facecolor('#fafafa')

    A_arr = np.nan_to_num(download_grid(erosion_imgs[yr], cuenca, scale=90), nan=0).squeeze()

    ax.imshow(hs_arr, cmap='Greys', extent=[lon_min, lon_max, lat_min, lat_max], alpha=0.30, zorder=1, vmin=0, vmax=255)
    masked_A = np.ma.masked_where(A_arr <= 0, A_arr)
    ax.imshow(masked_A, cmap=cmap, norm=norm, extent=[lon_min, lon_max, lat_min, lat_max], alpha=0.85, zorder=2,
              interpolation='nearest')

    for poly in cuenca_polys:
        ax.plot([pt[0] for pt in poly], [pt[1] for pt in poly], color='black', lw=1.8, zorder=4)

    A_mean_val = A_arr[A_arr > 0].mean() if (A_arr > 0).any() else 0
    A_max_val = A_arr.max()

    ax.set_title(f'Erosión Hídrica Proyectada — Año {yr}\n(Media: {A_mean_val:.2f} t/ha | Máx: {A_max_val:.1f} t/ha)',
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Longitud (°W)', fontsize=11)
    ax.set_ylabel('Latitud (°S)', fontsize=11)
    ax.tick_params(labelsize=10, direction='out')
    ax.set_aspect('equal', adjustable='box')

    ax.annotate('N', xy=(0.92, 0.94), xycoords='axes fraction', fontsize=13, fontweight='bold', ha='center',
                va='center')
    ax.annotate('', xy=(0.92, 0.93), xytext=(0.92, 0.83), xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=2.0))

    scale_km = 5
    x0 = lon_min + (lon_max - lon_min) * 0.05
    y0 = lat_min + (lat_max - lat_min) * 0.06
    scale_deg = scale_km / 111.0
    ax.plot([x0, x0 + scale_deg], [y0, y0], color='black', lw=2.4, zorder=5)
    ax.text(x0 + scale_deg / 2, y0 + 0.012, f'{scale_km} km', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.legend(handles=patches, title='Clase de erosión hídrica\n(t·ha⁻¹·año⁻¹)', loc='center left',
              bbox_to_anchor=(1.03, 0.5), frameon=True, edgecolor='#666', fontsize=11, title_fontsize=12)

    plt.tight_layout()
    register_figure(fig, f'fig18_mapa_erosion_{yr}', title=f'Mapa erosión {yr}', section='Erosión proyectada')
    plt.show()

### 8.2 Distribución de áreas y pérdida total año por año



In [ ]:
### 8.2 Distribución de áreas y pérdida total año por año
# === 8.2 Áreas por clase + tabla de pérdida anual ===
df_areas = pd.DataFrame(area_stats, index=CLASS_LABELS)
plot_years = sorted(area_stats.keys())

pct_por_año = df_areas[plot_years].div(df_areas[plot_years].sum(axis=0), axis=1) * 100
df_areas['% promedio'] = pct_por_año.mean(axis=1).round(2)
df_areas['Suma (ha)'] = df_areas[plot_years].sum(axis=1).round(2)

print('Áreas por clase de erosión (hectáreas):')
display(df_areas.round(2))

# Tabla de pérdida total basada en los puntos medios
class_midpoints = [10.0, 30.0, 50.0, 70.0, 90.0]
loss_records = []
for yr in plot_years:
    areas_yr = np.array([area_stats[yr][i] for i in range(5)])
    total_ha = areas_yr.sum()
    A_weighted = (areas_yr * np.array(class_midpoints)).sum() / total_ha if total_ha > 0 else 0
    loss_total_kt = (areas_yr * np.array(class_midpoints)).sum() / 1000
    R_yr = R_fut[R_fut['año'] == yr]['R'].values[0]
    loss_records.append({
        'Año': yr,
        'R (MJ·mm/ha·h·año)': round(R_yr, 1),
        'P_anual (mm)': round(df_forecast[df_forecast.año == yr].precip_pred.sum(), 1),
        'A_medio (t/ha/año)': round(A_weighted, 2),
        'Pérdida total (kt/año)': round(loss_total_kt, 1),
        'Área Alta+MuyAlta (ha)': round(areas_yr[3] + areas_yr[4], 1),
        '% crítica': round(100 * (areas_yr[3] + areas_yr[4]) / total_ha, 1) if total_ha > 0 else 0,
    })
df_loss_anual = pd.DataFrame(loss_records)

print('\nPérdida total año por año (mediana del ensemble):')
display(df_loss_anual)
register_table(df_loss_anual, 'tab04_perdida_anual', title='Tabla 4. Pérdida total', section='Erosión proyectada')

# =========================================================
# GRÁFICO APILADO (Corregido sin saturación de texto)
# =========================================================
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))  # Lienzo más grande

# (a) Área absoluta
bottoms = np.zeros(len(plot_years))
totals_ha = [sum([area_stats[y][i] for i in range(5)]) for y in plot_years]

for i, (lab, col) in enumerate(zip(CLASS_LABELS, RUSLE_COLORS)):
    vals = np.array([area_stats[y][i] for y in plot_years])
    axs[0].bar([str(y) for y in plot_years], vals, bottom=bottoms, color=col, label=lab, edgecolor='black', lw=0.6)

    # CORRECCIÓN: Solo se imprime el texto si la clase ocupa más del 4% del total
    for j, v in enumerate(vals):
        if v > 0.04 * totals_ha[j]:
            text_color = 'white' if col in ['#1a9641', '#d7191c', '#fdae61'] else 'black'
            axs[0].text(j, bottoms[j] + v / 2, f'{v:,.0f}', ha='center', va='center', fontsize=9, color=text_color,
                        fontweight='bold')
    bottoms += vals

axs[0].set_title('(a) Área absoluta por clase (ha)', fontweight='bold')
axs[0].set_ylabel('Área (ha)')
axs[0].set_xlabel('Año')

# (b) Porcentaje relativo
bottoms = np.zeros(len(plot_years))
for i, (lab, col) in enumerate(zip(CLASS_LABELS, RUSLE_COLORS)):
    vals = np.array(pct_por_año.loc[lab].values)
    axs[1].bar([str(y) for y in plot_years], vals, bottom=bottoms, color=col, label=lab, edgecolor='black', lw=0.6)

    # CORRECCIÓN: Solo se imprime el % si es mayor a 4.0%
    for j, v in enumerate(vals):
        if v > 4.0:
            text_color = 'white' if col in ['#1a9641', '#d7191c', '#fdae61'] else 'black'
            axs[1].text(j, bottoms[j] + v / 2, f'{v:.1f}%', ha='center', va='center', fontsize=9, color=text_color,
                        fontweight='bold')
    bottoms += vals

axs[1].set_title('(b) Composición relativa (%)', fontweight='bold')
axs[1].set_ylabel('Porcentaje del área total (%)')
axs[1].set_xlabel('Año')
axs[1].set_ylim(0, 100)

# Leyenda reposicionada debajo del gráfico para evitar superposición
handles = [mpatches.Patch(color=c, label=l) for c, l in zip(RUSLE_COLORS, CLASS_LABELS)]
fig.legend(handles=handles, title='Clase de erosión (t·ha⁻¹·año⁻¹)', loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.05), fontsize=11, title_fontsize=12)

plt.suptitle('Composición del territorio según clase de erosión proyectada', fontsize=15, fontweight='bold', y=0.96)
plt.tight_layout()
register_figure(fig, 'fig19_areas_erosion', title='Figura 19. Área por clase', section='Erosión proyectada')
plt.show()

---
## 9. Exportación de resultados


In [ ]:
!pip install weasyprint  pdfkit

In [ ]:
# ==============================================================================
# 9. EXPORTACIÓN DE RESULTADOS Y GENERACIÓN DE REPORTE HTML/PDF
# ==============================================================================
import os
import base64

out_dir = f'{WORK_DIR}/figures'
os.makedirs(out_dir, exist_ok=True)

# 9.1 Exportar CSVs
df.to_csv(f'{out_dir}/serie_unificada_1981_2025.csv', index=False)
df_forecast.to_csv(f'{out_dir}/pronostico_precipitacion_lurin.csv', index=False)
R_all.to_csv(f'{out_dir}/factor_R_anual.csv', index=False)
R_fut.to_csv(f'{out_dir}/factor_R_proyectado_ensemble.csv', index=False)
df_areas.to_csv(f'{out_dir}/areas_erosion_proyectada.csv')
df_loss_anual.to_csv(f'{out_dir}/perdida_anual.csv', index=False)
df_met.to_csv(f'{out_dir}/tabla_metricas.csv')

print('✅ Archivos CSV exportados correctamente en la carpeta figures.')

# 9.2 Generación Automatizada del Reporte HTML
print('⏳ Generando reporte consolidado...')

# Función para convertir imágenes a código Base64 (las incrusta dentro del archivo)
def image_to_base64(img_path):
    if os.path.exists(img_path):
        with open(img_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode()
            return f"data:image/png;base64,{encoded_string}"
    return ""

path_fig_obs = os.path.join(out_dir, 'fig11_obs_vs_pred.png')
path_fig_areas = os.path.join(out_dir, 'fig19_areas_erosion.png')
path_fig_mapas = os.path.join(out_dir, 'fig18_mapa_erosion_2030.png')

# Construcción de la plantilla HTML moderna
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Reporte Consolidado RUSLE</title>
    <style>
        body {{ font-family: 'Segoe UI', Helvetica, Arial, sans-serif; color: #333; line-height: 1.6; font-size: 11pt; max-width: 1000px; margin: auto; padding: 20px; }}
        h1 {{ color: #1f4e79; font-size: 22pt; text-align: center; border-bottom: 2px solid #1f4e79; padding-bottom: 10px; text-transform: uppercase; }}
        h2 {{ color: #2c3e50; font-size: 15pt; border-left: 5px solid #1f4e79; padding-left: 10px; margin-top: 40px; background-color: #f4f6f7; padding-top: 5px; padding-bottom: 5px; }}
        p {{ text-align: justify; margin-bottom: 15px; font-size: 11pt; }}
        table {{ width: 100%; border-collapse: collapse; margin-bottom: 30px; font-size: 10pt; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        th {{ background-color: #1f4e79; color: white; padding: 10px; text-align: center; border: 1px solid #1f4e79; }}
        td {{ padding: 8px; text-align: center; border: 1px solid #ddd; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .img-container {{ text-align: center; margin: 30px 0; }}
        .img-container img {{ max-width: 95%; border: 1px solid #ddd; border-radius: 6px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
        .header-sub {{ text-align:center; font-size:12pt; color:#555; font-weight:bold; margin-bottom: 30px; }}
    </style>
</head>
<body>
    <h1>Reporte Consolidado de Modelado Hidrológico y Erosión (RUSLE)</h1>
    <div class="header-sub">Cuenca del Río Lurín | Horizonte de Proyección: 2026 - 2030</div>

    <h2>1. Resumen de Métricas del Modelo LSTM</h2>
    <p>Evaluación de la eficiencia predictiva de las redes LSTM sobre el conjunto de entrenamiento y el conjunto de validación independiente.</p>
    {df_met.to_html(border=0, justify='center')}

    <div class="img-container">
        <img src="{image_to_base64(path_fig_obs)}" alt="Gráfico de Observado vs Predicho">
    </div>

    <h2>2. Estimación del Factor R de Erosividad</h2>
    <p>Valores proyectados del factor R (Fournier Modificado) a partir de la precipitación unificada pronosticada.</p>
    {R_fut[['año', 'P_total', 'R', 'R_p10', 'R_p90']].round(1).to_html(index=False, border=0, justify='center')}

    <h2>3. Dinámica y Pérdida de Suelo Proyectada</h2>
    <p>Estadísticas de pérdida total de suelo calculada mediante la integración espacial de los factores K, LS, C y P.</p>
    {df_loss_anual.to_html(index=False, border=0, justify='center')}

    <div class="img-container">
        <img src="{image_to_base64(path_fig_areas)}" alt="Gráfico de Distribución de Áreas">
    </div>

    <h2>4. Mapeo Espacial de Riesgo (Año Representativo: 2030)</h2>
    <p>Distribución geográfica de las zonas críticas basadas en la topografía D8 y la climatología proyectada.</p>
    <div class="img-container">
        <img src="{image_to_base64(path_fig_mapas)}" alt="Mapas de Erosión">
    </div>
</body>
</html>
"""

html_path = os.path.join(out_dir, 'Reporte_Final_RUSLE.html')

with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"📄 Reporte generado con éxito en: {html_path}")
print("   -> Para convertirlo a PDF: Abre este archivo en tu navegador web (Chrome/Edge) y usa la opción Imprimir -> 'Guardar como PDF'.")